# 🤗 Aula 12c: O Ecossistema Hugging Face e Transfer Learning com Transformers Pré-Treinados

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula12c_huggingface.ipynb)

> **Objetivos da Aula:**
> 1. Conhecer a biblioteca **Hugging Face `transformers`**, o padrão de fato da indústria moderna de inteligência artificial.
> 2. Compreender as vantagens de utilizar **modelos pré-treinados** (*Transfer Learning*) em bilhões de palavras em vez de treinar Transformers do zero com recursos limitados.
> 3. Utilizar a abstração de altíssimo nível **`pipeline`** para classificação de texto e análise de sentimentos.
> 4. Avaliar o modelo pré-treinado **`lvwerra/distilbert-imdb`** (uma versão destilada, 40% menor e 60% mais rápida do BERT) diretamente no conjunto de teste real do IMDB.
> 5. Comparar o tempo de inferência, custo computacional e acurácia (~93%+) em relação aos modelos treinados do zero.

## 1. Aquisição e Extração dos Dados do Stanford IMDB

> [!NOTE]
> Baixamos o conjunto original de 50.000 avaliações do IMDB e preparamos os diretórios locais para teste comparativo.

In [ ]:
# Faz o download do dataset oficial do IMDB comprimido
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
# Descompacta o arquivo tarball no ambiente de execução
!tar -xf aclImdb_v1.tar.gz
# Remove a pasta de dados não rotulados (unsup)
!rm -r aclImdb/train/unsup
# Exibe o conteúdo de um arquivo de crítica positiva como inspeção
!cat aclImdb/train/pos/4077_10.txt

## 2. Organização e Separação dos Subconjuntos de Dados

> [!NOTE]
> Realizamos a partição padrão criando a pasta `val/` com 20% das amostras de cada classe para manter os diretórios consistentes.

In [ ]:
# Importamos os módulos de manipulação de diretórios e aleatoriedade
import os, pathlib, shutil, random

# Definimos os caminhos dos diretórios
base_dir = pathlib.Path("aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"

# Iteramos pelas categorias para criar o split de validação
for category in ["pos", "neg"]:
  # Cria o diretório de destino correspondente se não existir
  os.makedirs(val_dir / category)
  # Lista os arquivos disponíveis na categoria
  files = os.listdir(train_dir / category)
  # Embaralha os arquivos de forma aleatória
  random.shuffle(files)
  # Define a fração de 20% para o conjunto de validação
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  # Move fisicamente os arquivos selecionados para o diretório de validação
  for fname in val_files:
    shutil.move(train_dir / category / fname, val_dir / category / fname)

## 3. Carregamento dos Datasets com `keras.utils.text_dataset_from_directory`

> [!TIP]
> Carregamos o conjunto de teste em lotes de 32 amostras para realizar inferência em lote com o modelo do Hugging Face.

In [ ]:
# Importamos a biblioteca Keras
from tensorflow import keras

# Definimos o tamanho dos lotes de streaming de dados
batch_size = 32
# Carregador de dados para treino
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
# Carregador de dados para validação
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
# Carregador de dados para o conjunto de teste (25.000 amostras)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

## 4. Pipeline de Vetorização de Texto do Keras (Referência)

> [!NOTE]
> Este bloco executa a vetorização e tokenização sequencial padrão no ecossistema TensorFlow/Keras para efeito de comparação com o tokenizador nativo do Hugging Face.

In [ ]:
# Importamos a camada TextVectorization do Keras
from tensorflow.keras.layers import TextVectorization

# Comprimento máximo fixo da sequência
max_length = 600
# Limite máximo de termos no vocabulário
max_tokens = 20000
# Instanciamos o vetorizador gerador de índices inteiros
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length)
# Extraímos apenas os textos para calcular as frequências de vocabulário
text_only_train_ds = train_ds.map(lambda x, y: x)
# Ajustamos o vetorizador ao corpus de treino
text_vectorization.adapt(text_only_train_ds)
# Mapeamos a tokenização paralelizada sobre o dataset de treino
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
# Mapeamos a tokenização paralelizada sobre o dataset de validação
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
# Mapeamos a tokenização paralelizada sobre o dataset de teste
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

# 5. Instalação e Inicialização da Biblioteca Hugging Face `transformers`

> [!IMPORTANT]
> A biblioteca **`transformers`** da Hugging Face fornece milhares de arquiteturas estado-da-arte (BERT, GPT-2/3/4, RoBERTa, T5, LLaMA) com pesos pré-treinados para PyTorch, TensorFlow e JAX.

In [ ]:
# Instala a versão mais recente da biblioteca transformers silenciosamente
!pip install transformers -qq

## 6. Criação da `pipeline` com o Modelo `lvwerra/distilbert-imdb`

> [!TIP]
> A função **`pipeline`** é a interface mais direta do Hugging Face. Ela empacota automaticamente:
> 1. O **Tokenizador** específico do modelo (WordPiece / Byte-Pair Encoding).
> 2. O **Modelo** com seus pesos pré-treinados transferidos do Hugging Face Hub.
> 3. O **Pós-processamento**, convertendo logits brutos em probabilidades calibradas e rótulos legíveis (*POSITIVE* / *NEGATIVE*).

In [ ]:
# Importamos a função utilitária pipeline de alto nível
from transformers import pipeline

# Instanciamos uma pipeline de classificação de texto apontando para o modelo DistilBERT ajustado no IMDB
pipe = pipeline("text-classification", model="lvwerra/distilbert-imdb")

## 7. Teste de Inferência Interativo com Frases Sintéticas

> [!NOTE]
> Passamos uma lista de strings em linguagem natural diretamente para a pipeline e inspecionamos o rótulo atribuído e o grau de confiança (*confidence score*).

In [ ]:
# Executamos a inferência da pipeline sobre duas sentenças com polaridades opostas
results = pipe(["This is a great movie.", "This is a bad movie."])
# Iteramos sobre os resultados retornados para cada texto
for result in results:
  # Exibimos o rótulo predito ('POSITIVE' ou 'NEGATIVE') e a probabilidade de confiança associada
  print(f"label: {result['label']}, with score: {result['score']:.4f}")

## 8. Avaliação de Desempenho no Conjunto de Teste Completo (25.000 Críticas)

> [!IMPORTANT]
> **Protocolo de Avaliação em Larga Escala:**
> - Percorremos o `test_ds`, decodificando cada texto de bytes para string UTF-8.
> - Truncamos os textos longos para no máximo 512 tokens (`truncation=True, max_length=512`), que é o limite nativo da atenção quadrática do DistilBERT.
> - Mapeamos o rótulo string (*POSITIVE* -> 1, *NEGATIVE* -> 0) e calculamos a acurácia global com `accuracy_score` e medimos o tempo total de execução com `time.time()`.

In [ ]:
# Importamos as métricas do Scikit-Learn e utilitários de sistema
from sklearn.metrics import accuracy_score
import numpy as np
import time

# Listas para armazenar predições binárias e rótulos reais de teste
y_pred = []
test_labels = []
# Registramos o instante de início para cronometrar a inferência
start = time.time()

# Iteramos por cada lote de críticas e rótulos do conjunto de teste
for batch, (text, targets) in enumerate(test_ds):
  # Iteramos por cada amostra individual dentro do lote
  for i in range(len(targets)):
    # Decodificamos o tensor binário em uma string de texto legível
    current_text = text[i].numpy().decode("utf-8")
    # Executamos a predição da pipeline com truncamento de segurança em 512 tokens
    result = pipe(current_text, truncation=True, max_length=512)
    # Extraímos o rótulo predito da lista de resultados
    label = result[0]["label"]
    # Convertemos o rótulo categórico para valor numérico binário
    if label == "POSITIVE":
      y_pred.append(1)
    else:
      y_pred.append(0)
  # Coletamos os rótulos reais do lote
  test_labels.extend(targets.numpy())
  # Imprimimos uma barra de progresso no console
  print(10*"\b", f"{batch}/{int(25_000/32)+1}")

# Exibimos o tempo total gasto na inferência em segundos
print(f"{time.time() - start}")
# Calculamos e exibimos a acurácia final sobre todo o conjunto de teste
accuracy_score(test_labels, y_pred)

## 9. Resumo da Aula e Conclusão do Módulo de Redes Neurais

### Comparativo Geral de Paradigmas no Dataset IMDB

| Abordagem | Arquitetura | Recursos Necessários | Acurácia no Teste | Velocidade de Treino |
| :--- | :--- | :--- | :---: | :---: |
| **NLP Clássico (Aula 10b)** | BoW / Bigramas / TF-IDF + MLP | CPU modesta | ~88.5% | Segundos |
| **Recorrência (Aula 10f/g)** | Embedding Treinável / GloVe + BiLSTM | GPU moderada | ~87.0% - 88.0% | Minutos |
| **Transformer do Zero (Aula 12a)** | TransformerEncoder + GlobalMaxPool | GPU recomendada | ~87.5% - 88.5% | Vários minutos |
| **Hugging Face Transfer Learning (Aula 12c)** | DistilBERT Pré-Treinado | GPU / CPU para inferência | **~93.0% - 94.0%+** | **Zero treino necessário** |

> [!IMPORTANT]
> **Parabéns! 🎓**
> Você concluiu com sucesso todo o módulo prático de **Redes Neurais Artificiais (ANN)**, percorrendo desde as equações matemáticas do Perceptron e Adaline (1958) até os Transformers pré-treinados e o estado-da-arte contemporâneo em Inteligência Artificial!